# NB4 · Analysis — Q1 to Q4

CPU only. Minutes, not hours. Re-run it as often as you like.

## Q1 is the whole point of this replication

Everything else here is secondary and would still be worth reporting, but the
question this project exists to answer is:

> On CIFAR-100, ViT and Mixer showed seed-reliability of **0.547** against
> **0.62–0.73** for every CNN. Does that survive at ImageNet scale, or was it a
> small-data artifact?

Q1 measures the **noise ceiling** ρ_seed: the Spearman correlation between the
per-sample MSC of two seeds of the *same* architecture. It is not a side
experiment — it is the denominator every transfer number gets divided by, and
it is the single most important quantity in the project.

## Read Q1 with the confound in mind

The eight architectures were trained for **equal epochs**, so schedule length is
not a variable — which it *was* on CIFAR (240 vs 300). But ViTs from scratch on
129k images will still land below the CNNs in accuracy, so **family and accuracy
remain partly confounded** and that must be stated wherever the result is.

The design carries three answers to it, and none of them is "the marginal means
look fine":

1. **`swin_tiny` vs `vit_small_p16`** — both attention; only Swin has locality
   and hierarchy. If reliability tracks *attention*, they agree. If it tracks
   *weak spatial prior*, Swin sits with the CNNs.
2. **`convnext_tiny` vs `resnet50`** — both convolution; only ConvNeXt uses the
   transformer design language.
3. **`vit_small_p16` vs `deit_small`** — **identical geometry, built by one
   function with one argument set**, differing only in augmentation strength.
   If ρ_seed differs across this pair, reliability is a property of *training*,
   not of attention — which would reframe the CIFAR finding rather than confirm
   it.

Together 1 and 2 form a 2×2: {conv, attention} × {strong prior, weak prior}. If
the effect is about attention the split runs along one diagonal; if it is about
spatial prior, the other.

## And one direct bridge

`shufflenetv2` is the only architecture measured in **both** studies. Its CIFAR
ρ_seed is **0.6698**. Whatever it reads here, the *difference* is a measurement
of what dataset scale alone does, with architecture held exactly fixed. It
calibrates every other comparison in the table.

In [ ]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    85d864742c05   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
    'ZXN0IGZvciB0aGUgb2ZmbGluZSBjaGVja3MiKQo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

In [ ]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

sess = M.Session(account='local', phase='p1', dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

---
## Q1 · Noise ceiling — ρ_seed per architecture

Reported as a curve over τ ∈ {0.0, 0.1, 0.2, 0.3, 0.5}. **If a conclusion holds
only at one τ, it is not a conclusion.** The pre-registered operating point is
τ = 0.1 and the pre-registered gate is ρ_seed ≥ 0.60.

In [ ]:
q1 = M.analyse_q1_all(sess)
M.save_analysis(sess.data_dir, 'q1_seed_ceilings_all', q1)
display(q1.sort_values('rho_seed_tau0.1', ascending=False))

In [ ]:
# The headline table: CNN vs non-CNN at tau=0.1, and the CIFAR comparison.
import numpy as np
col = 'rho_seed_tau0.1'
fam = {a: M.ZOO[a]['family'] for a in q1['arch']}
cnn = q1[q1['arch'].map(lambda a: fam[a] in ('resnet', 'vgg', 'mobile', 'convnext'))]
att = q1[q1['arch'].map(lambda a: fam[a] in ('vit', 'swin'))]

print(f"{'group':28s} {'n':>3s} {'range':>16s} {'mean':>8s}")
for name, g in (('convolutional', cnn), ('attention', att)):
    if len(g):
        print(f"{name:28s} {len(g):3d} "
              f"{g[col].min():.4f}-{g[col].max():.4f} {g[col].mean():8.4f}")

print()
print('CIFAR-100 was:  CNN 0.6217-0.7256 (mean 0.676) · ViT/Mixer 0.547')
print()
if len(cnn) and len(att):
    gap = cnn[col].min() - att[col].max()
    print(f'separation margin here: {gap:+.4f}   '
          f'({"clean, no overlap" if gap > 0 else "OVERLAPPING -- the CIFAR separation does NOT reproduce"})')
    print()
    print('Now check the confound before believing either answer:')
    sub = q1[['arch', col, 'top1_mean']].sort_values('top1_mean')
    display(sub)
    from scipy.stats import spearmanr
    if len(cnn) > 2:
        rho, p = spearmanr(cnn[col], cnn['top1_mean'])
        print(f'within CNNs, rho_seed vs top-1: Spearman {rho:+.3f} (p={p:.3f})')
        print('  near zero means accuracy carries little information about')
        print('  ceiling height INSIDE a family -- which is the argument that')
        print('  the family effect is not an accuracy effect.')

In [ ]:
# The three internal controls. These do not depend on the marginal means.
for a, b, what in (('swin_tiny', 'vit_small_p16', 'spatial prior, attention held fixed'),
                   ('convnext_tiny', 'resnet50', 'design language, convolution held fixed'),
                   ('deit_small', 'vit_small_p16', 'RECIPE, geometry held fixed')):
    ra = q1.loc[q1.arch == a, col]
    rb = q1.loc[q1.arch == b, col]
    if len(ra) and len(rb):
        print(f'{a:15s} {float(ra.iloc[0]):.4f}   vs   {b:15s} {float(rb.iloc[0]):.4f}'
              f'   d={float(ra.iloc[0]) - float(rb.iloc[0]):+.4f}   [{what}]')

print()
print('shufflenetv2 -- the only architecture in BOTH studies:')
r = q1.loc[q1.arch == 'shufflenetv2_in', col]
if len(r):
    print(f'  ImageNet-100 {float(r.iloc[0]):.4f}   CIFAR-100 0.6698   '
          f'd={float(r.iloc[0]) - 0.6698:+.4f}')
    print('  That difference is what dataset scale does with architecture')
    print('  held exactly fixed. It calibrates every row above.')

---
## Q2 · Is compute-need one-dimensional across axes?

PCA over per-sample MSC on {depth, resolution-proxy, precision}. H2 predicted
PC1 ≥ 0.60. On CIFAR **0 of 15** architectures reached it and the highest
anywhere was 0.532 — not a marginal miss.

In [ ]:
q2 = M.analyse_q2_all(sess)
M.save_analysis(sess.data_dir, 'q2_axis_structure_all', q2)
display(q2.sort_values('pc1', ascending=False))
print(f"reaching PC1 >= 0.60: {int((q2['pc1'] >= 0.60).sum())} of {len(q2)}")

---
## Q3 · Transfer across architectures

The disattenuated transfer coefficient, T = ρ(A,B) / √(ρ_seed(A)·ρ_seed(B)).
Dividing by the ceilings is what turns "0.65 seems highish?" into a defensible
claim — and it is the correction the example-difficulty literature generally
omits.

**The shuffled control runs first.** It compares the raw correlation against the
exact permutation null 1/√(n−1), requires both |z| > 5 **and** |ρ| > 0.10, and
takes the worst of three permutations. An earlier version used a bare
`|T| < 0.05` threshold, which was sample-size blind, ceiling-dependent in the
worst direction (≈7× more likely to false-alarm on exactly the low-ceiling ViT
pairs carrying the headline), and two-sided against a one-sided failure mode. It
halted the analysis on a perfectly healthy pair.

In [ ]:
ctrl = M.analyse_q3_shuffled_control_all(sess)
M.save_analysis(sess.data_dir, 'q3_shuffled_control', ctrl)
bad = ctrl[~ctrl['passed']]
print(f"{len(ctrl) - len(bad)}/{len(ctrl)} shuffled controls pass  "
      f"(max |z| = {ctrl['z'].abs().max():.2f} against a 5-sigma threshold)")
if len(bad):
    display(bad)
    print('*** Tables may be misaligned. This is a BUG, not a finding.')

In [ ]:
q3 = M.analyse_q3_all(sess)
M.save_analysis(sess.data_dir, 'q3_transfer_matrix', q3)
print(q3.groupby('pair_type')['T'].agg(['count', 'mean', 'std', 'min', 'max']))

---
## Q4 · Is MSC reducible to classical difficulty scores?

Nested-model ΔR² against the full **seven**-score battery
(`msp, margin, entropy, ce_loss, el2n, forget_events, pred_depth`), on
`train_holdout` — the only split where EL2N and forgetting-events are defined.

Running this on the test split with five of seven scores handicaps the battery,
which flatters MSC. On CIFAR that overstated irreducibility by **2.5×** and the
number had to be withdrawn.

In [ ]:
q4 = M.analyse_q4_all(sess, split='train_holdout')
M.save_analysis(sess.data_dir, 'q4_irreducibility_all', q4)
print(f"median delta-R2 {q4['delta_r2'].median():.4f}   "
      f"clearing 0.05: {int((q4['delta_r2'] >= 0.05).sum())}/{len(q4)}")
print(f"median partial rho {q4['partial_spearman'].median():.4f}   gate 0.30")
print()
print('Split CNN-only vs transformer-involving before reading either number.')
print('A noisier measurement necessarily explains less variance, so a low')
print('transformer delta-R2 is NOT an independent finding from a low Q1')
print('ceiling -- report them together or a reader double-counts them.')

---
## Paper outputs

Every contribution the protocol claims has to be backed by an artifact on disk,
or it is a claim and not a result. This cell writes them and then **checks the
list**, so a missing table is reported rather than discovered while writing.

| # | contribution (protocol §8.1) | artifact |
|---|---|---|
| 1 | MSC: per-sample, cost-normalised, multi-axis, stability-closed | `runs/*/per_sample/*.parquet` + `budgets/*.json` |
| 2 | first measurement of whether compute-need is one-dimensional across axes | `analysis/q2_axis_structure_all.csv`, Table 3 |
| 3 | first noise-ceiling-corrected cross-architecture transfer study | `analysis/q1_seed_ceilings_all.csv`, `q3_transfer_matrix.csv`, Tables 2 and 4 |
| 4 | irreducibility to seven classical difficulty scores | `analysis/q4_irreducibility_all.csv`, Table 5 |
| 5 | MSC-KD, benchmarked at matched FLOPs | NB5 → `analysis/q5_method_comparison.csv` |
| 6 | fully reproducible artifact | `paper/provenance.csv`, `tables/`, every config and log |

### The one this replication adds

**Contribution 3 is where the novelty concentrates**, and it is sharper here
than on CIFAR. The methodological point is that *measurement reliability is
itself architecture-dependent*, so a cross-architecture difficulty study that
does not disattenuate is comparing quantities measured with unequal precision —
and the example-difficulty literature generally does not.

CIFAR demonstrated that. This tests whether it **survives a 40× increase in
dataset size and a 49× increase in pixels**, with four independent crossings of
the CNN/attention boundary and one architecture held fixed across both studies.
Either answer is a result; the second is a self-retraction, which is rarer and
more useful than the first.

In [ ]:
from pathlib import Path
import pandas as pd

tables = Path(sess.data_dir) / 'tables'
tables.mkdir(parents=True, exist_ok=True)

# Table 1 -- the atlas: what was trained, and did it converge.
rows = []
for r in sess.completed_runs(phase='p1'):
    s = M.read_json(M.run_layout(sess.work, r['run_id'])['base'] / 'summary.json', {})
    if not s:
        continue
    m = M.parse_run_id(r['run_id'])
    rows.append({'arch': m['arch'], 'family': M.ZOO.get(m['arch'], {}).get('family'),
                 'seed': m['seed'], 'top1': s.get('best_accuracy'),
                 'epochs': s.get('num_epochs_run'),
                 'params_M': (s.get('num_parameters') or 0) / 1e6,
                 'gflops': (s.get('full_flops') or 0) / 1e9,
                 'gpu_hours': (s.get('total_time_sec') or 0) / 3600,
                 'kwh': s.get('total_energy_kwh'),
                 'measured': sess.measured(r['run_id'])})
t1 = pd.DataFrame(rows)
t1.to_csv(tables / 'table1_atlas.csv', index=False)
display(t1)

# Table 2 -- Q1, the headline. rho_seed beside accuracy, because the confound
# has to be visible in the same table rather than argued around afterwards.
t2 = q1[['arch', 'family', 'n_seeds', 'n_pairs', 'top1_mean', 'top1_spread',
         'rho_seed_tau0.1', 'rho_seed_sd_tau0.1', 'j10_tau0.1']].copy()
t2 = t2.sort_values('rho_seed_tau0.1', ascending=False)
t2.to_csv(tables / 'table2_q1_ceilings.csv', index=False)
display(t2)

In [ ]:
# Table 3 (Q2), Table 4 (Q3), Table 5 (Q4)
q2.to_csv(tables / 'table3_q2_axis_structure.csv', index=False)
q3.to_csv(tables / 'table4_q3_transfer.csv', index=False)
q4.to_csv(tables / 'table5_q4_irreducibility.csv', index=False)

# Table 6 -- the CIFAR<->ImageNet comparison. This table IS the paper.
CIFAR = {'shufflenetv2': 0.6698, 'vit_tiny': 0.5475, 'mixer_nano': 0.5470,
         'convnext_femto': 0.7084, 'resnet32x4': 0.7256, 'vgg8': 0.7216}
comp = []
for _, r in q1.iterrows():
    prior = CIFAR.get(M.CROSS_STUDY_ALIAS.get(r['arch'], r['arch']))
    comp.append({'arch': r['arch'], 'family': r['family'],
                 'in100_rho_seed': r['rho_seed_tau0.1'],
                 'cifar_rho_seed': prior,
                 'delta': (r['rho_seed_tau0.1'] - prior) if prior else None,
                 'same_architecture': prior is not None
                                      and r['arch'] in M.CROSS_STUDY_ALIAS})
t6 = pd.DataFrame(comp)
t6.to_csv(tables / 'table6_cifar_vs_imagenet.csv', index=False)
display(t6)
print()
print('Only the row with same_architecture=True is a controlled comparison.')
print('The others differ in architecture AND scale, so their delta mixes two')
print('effects and cannot be read as "what scale did".')

In [ ]:
# Figures. Small, because a paper needs few and each has to earn its place.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

figs = Path(sess.data_dir) / 'paper' / 'figures'
figs.mkdir(parents=True, exist_ok=True)

# Fig 1 -- rho_seed by architecture, coloured by family, with the CIFAR band.
fig, ax = plt.subplots(figsize=(7, 4))
d = q1.sort_values('rho_seed_tau0.1')
cols = ['tab:red' if f in ('vit', 'swin') else 'tab:blue' for f in d['family']]
ax.barh(d['arch'], d['rho_seed_tau0.1'], color=cols)
ax.axvline(0.60, ls='--', c='k', lw=1, label='pre-registered gate 0.60')
ax.axvspan(0.6217, 0.7256, alpha=0.10, color='tab:blue', label='CIFAR CNN band')
ax.axvspan(0.5470, 0.5475, alpha=0.25, color='tab:red', label='CIFAR ViT/Mixer')
ax.set_xlabel(r'$\rho_{seed}$  ($\tau$=0.1, depth axis)')
ax.legend(fontsize=7)
fig.tight_layout()
M.save_figure(fig, sess.data_dir, 'fig1_q1_ceilings')

# Fig 2 -- the tau curve. No conclusion may depend on tau, so show it.
fig, ax = plt.subplots(figsize=(7, 4))
taus = [0.0, 0.1, 0.2, 0.3, 0.5]
for _, r in q1.iterrows():
    c = 'tab:red' if r['family'] in ('vit', 'swin') else 'tab:blue'
    ax.plot(taus, [r.get(f'rho_seed_tau{t}') for t in taus], marker='o',
            color=c, alpha=0.7, label=r['arch'])
ax.set_xlabel(r'$\tau$'); ax.set_ylabel(r'$\rho_{seed}$')
ax.legend(fontsize=6, ncol=2)
fig.tight_layout()
M.save_figure(fig, sess.data_dir, 'fig2_tau_curves')

# Fig 3 -- the confound, plotted rather than asserted.
fig, ax = plt.subplots(figsize=(5, 4))
for _, r in q1.iterrows():
    c = 'tab:red' if r['family'] in ('vit', 'swin') else 'tab:blue'
    ax.scatter(r['top1_mean'], r['rho_seed_tau0.1'], color=c)
    ax.annotate(r['arch'], (r['top1_mean'], r['rho_seed_tau0.1']), fontsize=6)
ax.set_xlabel('top-1 (%)'); ax.set_ylabel(r'$\rho_{seed}$')
ax.set_title('the confound, shown')
fig.tight_layout()
M.save_figure(fig, sess.data_dir, 'fig3_ceiling_vs_accuracy')
print('figures written to paper/figures/')

In [ ]:
M.provenance_manifest(sess.data_dir)

# Check the list rather than trusting it. A missing table found here costs a
# re-run of a CPU notebook; found while writing, it costs a day.
rep = M.verify_paper_artifacts(sess.data_dir)
for r in rep['rows']:
    print(f"  [{r['state']:7s}] {r['artifact']:46s} {r['backs']}")

print()
if not rep['ok']:
    print(f"  *** {len(rep['missing'])} paper artifact(s) absent. The")
    print(f"  *** contributions they back are claims, not results.")
else:
    print('  every claimed contribution has an artifact behind it.')
    print()
    print('  Q5 (the method) needs NB5. Q1-Q4 stand without it -- that')
    print('  separation is the point of the protocol restructure.')